In [1]:
import pandas as pd
import os
import warnings
from pandasql import sqldf
from datetime import datetime, timedelta
import glob
warnings.filterwarnings("ignore")

### Nielsen

In [2]:
dir = os.getcwd()
# Important!! Make sure the file exist and refreshed first
xls = pd.ExcelFile(f'{dir}/../Data Source/Nielsen/Nielsen Brand Ranking_Jun26 v270726.xlsx')
sheet_names = ['SG Female CPD', 'SG Male CPD']
dfs = {}
# Read each sheet into a DataFrame and store it in the dictionary
for sheet_name in sheet_names:
    dfs[sheet_name] = pd.read_excel(xls, sheet_name=sheet_name)

In [3]:
for df in dfs:
    print(df)

SG Female CPD
SG Male CPD


In [4]:
# Brand Mapping
def map_brand(row):
    if pd.notna(row['LOCAL BRAND']):
        return row['LOCAL BRAND']
    else:
        return row['LOCAL MANUF']

In [5]:
# Get Current Year, Last Year and Year before last year
current_year = 2026
last_year = current_year - 1
year_before_last_year = current_year - 2
print(current_year, last_year, year_before_last_year)

2026 2025 2024


In [6]:
# Year Transformation
def map_year(row):
    if 'Cal Yr' in row['Periods']:
        year = pd.Series(row['Periods']).str.extract(r'(\d{4})').values[0]
        return year.astype(str)[0]
    
    elif 'YTD YA' in row['Periods']:
            return 'YTD ' + str(last_year)
    
    elif 'YTD' in row['Periods']:
            return 'YTD ' + str(current_year)
    

In [7]:
two_digit_year =  dfs['SG Female CPD']['Periods'].str.extract(r'(\d{2}$)').values[0]
two_digit_year
print('20' + two_digit_year.astype(str)[0])

2025


In [8]:
# Axes Category Mapping
def map_axes(row):
    if ('GLOBAL SEGMENT' in row.index) and (row['GLOBAL SEGMENT'] == 'FEMALE'):
        return 'Female Skincare'
    elif ('GLOBAL SEGMENT' in row.index) and (row['GLOBAL SEGMENT'] == 'MALE'):
        return 'Male Skincare'

In [9]:
# Important!! Try to understand the filter and logic here
for df in dfs:
    dfs[df]['Year'] = dfs[df].apply(map_year, axis=1)
    dfs[df]['Mass/Mass medic'] = 'Mass'
    dfs[df]['Brand'] = dfs[df].apply(map_brand, axis=1)
    dfs[df]['Axes'] = dfs[df].apply(map_axes, axis=1)

In [10]:
dfs['SG Female CPD']['Periods'].unique()

<StringArray>
[  'YTD YA - 26 w/e 29/06/25',      'YTD - 26 w/e 28/06/26',
 'Cal Yr 2024 - w/e 29/12/24', 'Cal Yr 2025 - w/e 28/12/25']
Length: 4, dtype: str

In [11]:
dfs['SG Female CPD']

,Markets,Periods,SECTOR,GLOBAL SEGMENT,LOCAL MANUF,LOCAL BRAND,Sales Value,Year,Mass/Mass medic,Brand,Axes
0,Modern Trade/Singapore,YTD YA - 26 w/e 29/06/25,MASS,FEMALE,TOTAL OTHERS,NaN,2536245.424,YTD 2025,Mass,TOTAL OTHERS,Female Skincare
1,Modern Trade/Singapore,YTD YA - 26 w/e 29/06/25,MASS,FEMALE,NaN,100% PURE,134.130,YTD 2025,Mass,100% PURE,Female Skincare
2,Modern Trade/Singapore,YTD YA - 26 w/e 29/06/25,MASS,FEMALE,NaN,1028 VISUAL THERAPY,235.400,YTD 2025,Mass,1028 VISUAL THERAPY,Female Skincare
3,Modern Trade/Singapore,YTD YA - 26 w/e 29/06/25,MASS,FEMALE,NaN,3W CLINIC,468.150,YTD 2025,Mass,3W CLINIC,Female Skincare
4,Modern Trade/Singapore,YTD YA - 26 w/e 29/06/25,MASS,FEMALE,NaN,9 WISHES,877.450,YTD 2025,Mass,9 WISHES,Female Skincare
...,...,...,...,...,...,...,...,...,...,...,...
910,Modern Trade/Singapore,Cal Yr 2025 - w/e 28/12/25,MASS,FEMALE,NaN,WOW,1012.780,2025,Mass,WOW,Female Skincare
911,Modern Trade/Singapore,Cal Yr 2025 - w/e 28/12/25,MASS,FEMALE,NaN,YAMANO,588.600,2025,Mass,YAMANO,Female Skincare
912,Modern Trade/Singapore,Cal Yr 2025 - w/e 28/12/25,MASS,FEMALE,NaN,YUKAZAN,4809.960,2025,Mass,YUKAZAN,Female Skincare
913,Modern Trade/Singapore,Cal Yr 2025 - w/e 28/12/25,MASS,FEMALE,NaN,ZAPPY,34385.460,2025,Mass,ZAPPY,Female Skincare


In [12]:
print(dfs[df].columns)

Index(['Markets', 'Periods', 'SECTOR', 'GLOBAL SEGMENT', 'LOCAL MANUF',
       'LOCAL BRAND', 'Sales Value', 'Year', 'Mass/Mass medic', 'Brand',
       'Axes'],
      dtype='str')


In [13]:
for df in dfs:
    # Pivot
    dfs[df] = dfs[df].pivot_table(
        index=['Mass/Mass medic', 'Axes', 'Brand'],
        columns='Year',
        values='Sales Value',
        aggfunc='sum'
    )

    dfs[df] = dfs[df].reset_index()

    # Get available columns
    cols = dfs[df].columns.tolist()

    # Define desired columns
    desired_cols = [
        'Mass/Mass medic',
        'Axes',
        'Brand',
        str(year_before_last_year),   # e.g. 2023
        str(last_year),               # e.g. 2025
        f'YTD {last_year}',           # YTD 2025
        f'YTD {current_year}'         # YTD 2026
    ]

    # Keep only columns that actually exist (prevents KeyError)
    final_cols = [col for col in desired_cols if col in cols]

    dfs[df] = dfs[df][final_cols]

    # Fill missing values
    dfs[df] = dfs[df].fillna(0)

    # Optional: sort columns nicely (YTD at the end)
    non_ytd = [col for col in final_cols if 'YTD' not in col and col not in ['Mass/Mass medic', 'Axes', 'Brand']]
    ytd = [col for col in final_cols if 'YTD' in col]

    dfs[df] = dfs[df][['Mass/Mass medic', 'Axes', 'Brand'] + sorted(non_ytd) + sorted(ytd)]

    # Preview
    print(f"\nProcessed: {df}")
    display(dfs[df].head(3))


Processed: SG Female CPD


Year,Mass/Mass medic,Axes,Brand,2024,2025,YTD 2025,YTD 2026
0,Mass,Female Skincare,100% PURE,211.63,134.13,134.13,0.00
1,Mass,Female Skincare,1028 VISUAL THERAPY,6126.77,235.40,235.40,0.00
2,Mass,Female Skincare,3W CLINIC,425.17,679.30,468.15,297.65



Processed: SG Male CPD


Year,Mass/Mass medic,Axes,Brand,2024,2025,YTD 2025,YTD 2026
0,Mass,Male Skincare,&SONS,9376.30,2798.69,2111.37,123.63
1,Mass,Male Skincare,BAD LAB,139.04,143.60,92.26,124.80
2,Mass,Male Skincare,BIELENDA,12.90,0.00,0.00,0.00


In [14]:
nielsen = pd.concat(dfs.values())
# Data Checking
nielsen.head()

Year,Mass/Mass medic,Axes,Brand,2024,2025,YTD 2025,YTD 2026
0,Mass,Female Skincare,100% PURE,211.630,134.130,134.130,0.00
1,Mass,Female Skincare,1028 VISUAL THERAPY,6126.770,235.400,235.400,0.00
2,Mass,Female Skincare,3W CLINIC,425.170,679.300,468.150,297.65
3,Mass,Female Skincare,9 WISHES,2672.980,2722.360,877.450,421.04
4,Mass,Female Skincare,ABIB,577436.505,615039.735,340359.765,261578.39


### OMT

In [15]:
base_dir = 'C:\\Users\\balatarsini_avinitya\\Downloads\\CPD & LDB O+O - JUNE\\loreal-report-automation (2)\\Brand Ranking'

In [16]:
# Important!! Make sure the files exist and are updated first
omt_dir = os.path.join(base_dir, '..\\Data Source\\OMT - O+O\\SG CPD')
omt_start_month = pd.Period('2024-01', freq='M')
omt_pattern = os.path.join(omt_dir, 'OMT SG CPD *.xlsx')
available_omt_months = []

for file in glob.glob(omt_pattern):
    month_text = os.path.splitext(os.path.basename(file))[0].replace('OMT SG CPD ', '')
    try:
        available_omt_months.append(pd.Period(month_text, freq='M'))
    except ValueError:
        pass

if not available_omt_months:
    raise FileNotFoundError(f'No OMT files found in {omt_dir}')

omt_end_month = max(available_omt_months)
omt_months = pd.period_range(omt_start_month, omt_end_month, freq='M')
print(f'Latest OMT month found: {omt_end_month}')
omt_files = [os.path.join(omt_dir, f'OMT SG CPD {month}.xlsx') for month in omt_months]

missing_files = [file for file in omt_files if not os.path.exists(file)]
if missing_files:
    raise FileNotFoundError('Missing OMT files:\n' + '\n'.join(missing_files))

omt_raw = pd.concat([pd.read_excel(file, sheet_name='Export', keep_default_na=False) for file in omt_files], axis=0)

# Rename the literal "na" brand
omt_raw['Brand'] = omt_raw['Brand'].apply(
    lambda x: 'NA Brand'
    if isinstance(x, str) and x.strip().lower() == 'na'
    else x
)


omt_wg = omt_raw.copy()
omt_f = omt_raw.copy()

mapping = pd.read_excel(f'{dir}/../Data Source/CPD Skincare Mapping/Skincare Mapping.xlsx', sheet_name='SG')

Latest OMT month found: 2026-06


In [17]:
# Structure the df into dictionary
omt = {
    'omt_wg': omt_wg,
    'omt_f': omt_f
}

In [18]:
# Brand Mapping
def map_brand_group(row):
    if row['Brand'] in ["GARNIER", "MAYBELLINE","3CE"]:
        return row['Brand']
    elif row['Brand'] == "L'OREAL PARIS":
        return 'LOREAL PARIS'
    else:
        return 'Market'

In [19]:
# Important!! Try to understand the filter and logic here
for df in omt:
    omt[df] = omt[df][omt[df]['Category L1'] != 'FRAGRANCE']
    omt[df] = omt[df][omt[df]['Category L2'].isin(['EYE MAKEUP','FACE MAKEUP','LIP MAKEUP', 'NAIL MAKEUP', 'OTHER MAKEUP','FACE CARE & CLEANSING','SUN CARE','HAIR COLOR','HAIR CARE'])]
    omt[df] = omt[df][(omt[df]['Category L2'] != 'SUN CARE') | (omt[df]['Category L3'] == 'FACE PROTECTION')]
    omt[df][['Year', 'Month']] = omt[df]['Year Month'].str.split('-', expand=True)
    omt[df] = omt[df].dropna(subset=['Year'])
    omt[df]['Subdivision'] = 'NA'
    omt[df][['Year', 'Month']] = omt[df][['Year', 'Month']].astype(int)
    omt[df]['brand_group'] = omt[df].apply(map_brand_group, axis=1)
    omt[df] = omt[df][['Mall Type', 'brand_group', 'Brand', 'Year', 'Month', 'Universe', 'Subdivision', 'Category L1', 'Category L2','Total Est. Sales Local']]
    omt[df].reset_index(drop=True, inplace=True)
    omt[df] = omt[df].groupby(['Mall Type', 'brand_group', 'Brand', 'Year', 'Month', 'Universe', 'Subdivision', 'Category L1', 'Category L2'])[[ 'Total Est. Sales Local']].sum().reset_index()
    # test for 2023 first
    omt[df].reset_index(drop=True, inplace=True)
    omt[df] = omt[df].sort_values(by=['Year', 'Month'])

    omt[df] = omt[df][omt[df]['Year'] >= 2022]

#### OMT Skincare

In [20]:
# Skincare Percentage Share for Female and Male
omt_wg_sc = omt['omt_wg'][omt['omt_wg']['Category L1'] == 'SKIN CARE']
omt_f_sc = omt['omt_f'][omt['omt_f']['Category L1'] == 'SKIN CARE']
omt_sc = {
    'omt_wg_sc': omt_wg_sc,
    'omt_f_sc': omt_f_sc
}

In [21]:
omt_wg_sc['Brand'].unique()

<StringArray>
[                   'GARNIER',              'L'OREAL PARIS',
                 'MAYBELLINE',                     '&HONEY',
                 '100 % PURE',        '1028 VISUAL THERAPY',
                   '111 SKIN',                         '3M',
                  '3W CLINIC',                      'A'KIN',
 ...
                        'CMD',                      'COKKI',
                   'DEEPONDE',                    'EST.LAB',
                    'FREBITS', 'FRESH LOTUS YOUTH PRESERVE',
                     'GROWUS',                   'POSTCARD',
                      'T O N',                 'WEST&MONTH']
Length: 2434, dtype: str

In [22]:
# Seperate wg and f
omt_wg_sc = omt_sc['omt_wg_sc']
omt_f_sc = omt_sc['omt_f_sc']

In [23]:
# Join with Mapping sheet using SQL query
query_1 = f"""
        WITH mapping_tx AS (
            SELECT 
                Brand,
                Year,
                Month,
                Shopee,
                Lazada,
                'FEMALE SKINCARE' AS Category
            FROM mapping
            
            UNION
        
            SELECT
                Brand,
                Year,
                Month,
                1 - Shopee AS Shopee,
                1 - Lazada AS Lazada,
                'MALE SKINCARE' AS Category
            FROM mapping
        )

        SELECT
            a.Brand,
            a.Year,
            a.Month,
            a.Universe,
            a.Subdivision,
            b.Category AS "Category L1",
            a."Category L2",
          
            CASE
                WHEN a."Mall Type" = 'Shopee Mall' THEN a."Total Est. Sales Local" * b.Shopee
                WHEN a."Mall Type" = 'Lazada Mall' THEN a."Total Est. Sales Local" * b.Lazada
            END AS "Total Est. Sales Local"
        FROM omt_wg_sc a
            LEFT JOIN mapping_tx b
                ON (
                    a.brand_group = b.Brand
                    AND a.Year = b.Year
                    AND a.Month = b.Month
                )
        """

In [24]:
# Join with Mapping sheet using SQL query
query_2 = f"""
        WITH mapping_tx AS (
            SELECT 
                Brand,
                Year,
                Month,
                Shopee,
                Lazada,
                'FEMALE SKINCARE' AS Category
            FROM mapping
        
            UNION
        
            SELECT
                Brand,
                Year,
                Month,
                1 - Shopee AS Shopee,
                1 - Lazada AS Lazada,
                'MALE SKINCARE' AS Category
            FROM mapping
        )
        
        SELECT
            a.Brand,
            a.Year,
            a.Month,
            a.Universe,
            a.Subdivision,
            b.Category AS "Category L1",
            a."Category L2",
        
            CASE
                WHEN a."Mall Type" = 'Shopee Mall' THEN a."Total Est. Sales Local" * b.Shopee
                WHEN a."Mall Type" = 'Lazada Mall' THEN a."Total Est. Sales Local" * b.Lazada
            END AS "Total Est. Sales Local"
        FROM omt_f_sc a
            LEFT JOIN mapping_tx b
                ON (
                    a.brand_group = b.Brand
                    AND a.Year = b.Year
                    AND a.Month = b.Month
                )
        """

In [25]:
# run the query and store the result in a new dataframe
omt_wg_sc = sqldf(query_1)
omt_f_sc = sqldf(query_2)

In [26]:
omt_wg_sc.tail(5)

,Brand,Year,Month,Universe,Subdivision,Category L1,Category L2,Total Est. Sales Local
97913,ÉST.LAB,2026,6,MASS,NA,FEMALE SKINCARE,FACE CARE & CLEANSING,2277.873230
97914,ÉST.LAB,2026,6,MASS,NA,MALE SKINCARE,SUN CARE,47.406569
97915,ÉST.LAB,2026,6,MASS,NA,FEMALE SKINCARE,SUN CARE,2341.073431
97916,彩棠 TIMAGE,2026,6,MASS,NA,MALE SKINCARE,FACE CARE & CLEANSING,0.315583
97917,彩棠 TIMAGE,2026,6,MASS,NA,FEMALE SKINCARE,FACE CARE & CLEANSING,15.584417


In [27]:
omt_f_sc.tail(5)

,Brand,Year,Month,Universe,Subdivision,Category L1,Category L2,Total Est. Sales Local
97913,ÉST.LAB,2026,6,MASS,NA,FEMALE SKINCARE,FACE CARE & CLEANSING,2277.873230
97914,ÉST.LAB,2026,6,MASS,NA,MALE SKINCARE,SUN CARE,47.406569
97915,ÉST.LAB,2026,6,MASS,NA,FEMALE SKINCARE,SUN CARE,2341.073431
97916,彩棠 TIMAGE,2026,6,MASS,NA,MALE SKINCARE,FACE CARE & CLEANSING,0.315583
97917,彩棠 TIMAGE,2026,6,MASS,NA,FEMALE SKINCARE,FACE CARE & CLEANSING,15.584417


#### OMT Hair & Makeup

In [28]:
# Select the columns for Hair and Makeup
for df in omt:
    omt[df] = omt[df][['Brand', 'Year', 'Month', 'Universe', 'Subdivision', 'Category L1', 'Category L2','Total Est. Sales Local']]
    omt[df].reset_index(drop=True, inplace=True)

In [29]:
# Exclude Skincare
omt_wg_hm = omt['omt_wg'][omt['omt_wg']['Category L1'] != 'SKIN CARE']
omt_f_hm = omt['omt_f'][omt['omt_f']['Category L1'] != 'SKIN CARE']

In [30]:
omt_f_hm

,Brand,Year,Month,Universe,Subdivision,Category L1,Category L2,Total Est. Sales Local
0,3CE,2024,1,MASS,NA,MAKEUP,EYE MAKEUP,4285.45
1,3CE,2024,1,MASS,NA,MAKEUP,FACE MAKEUP,5130.7
2,3CE,2024,1,MASS,NA,MAKEUP,LIP MAKEUP,7964.09
3,3CE,2024,1,MASS,NA,MAKEUP,OTHER MAKEUP,78.99
4,GARNIER,2024,1,MASS,NA,HAIR,HAIR CARE,51.4
...,...,...,...,...,...,...,...,...
105536,ZIAJA,2026,6,MASS,NA,HAIR,HAIR CARE,50.6
105540,彩棠 TIMAGE,2026,6,MASS,NA,MAKEUP,EYE MAKEUP,43.56
105541,彩棠 TIMAGE,2026,6,MASS,NA,MAKEUP,FACE MAKEUP,1930.42
105542,彩棠 TIMAGE,2026,6,MASS,NA,MAKEUP,LIP MAKEUP,22.79


#### Merge Skincare and Hair and Makeup

In [31]:
omt_wg = pd.concat([omt_wg_sc, omt_wg_hm], ignore_index=True)
omt_f = pd.concat([omt_f_sc, omt_f_hm], ignore_index=True)

### Seperating Data into Different Tabs

In [32]:
# Get last month
last_month = datetime.now().replace(day=1) - timedelta(days=1)
month_abbr = last_month.strftime("%b").upper()
year = last_month.year

# Format the output as "MMM YYYY"
# filemonth = last_month.strftime("%b %Y").upper()
filemonth = "JUNE 2026"
# Print the result
print(f"{filemonth}")

JUNE 2026


In [33]:
if not os.path.exists(f'../Generated Data/Brand Ranking/{filemonth}'):
        os.makedirs(f'../Generated Data/Brand Ranking/{filemonth}')

In [34]:
with pd.ExcelWriter(f'../Generated Data/Brand Ranking/{filemonth}/SG CPD Brand Ranking {filemonth}.xlsx', engine='xlsxwriter') as writer:
    nielsen.to_excel(writer, sheet_name='Nielsen', index=False)
    omt_wg.to_excel(writer, sheet_name='OMT-WG', index=False)
    omt_f.to_excel(writer, sheet_name='OMT-F', index=False)